In [ ]:
# Ingestão de Subtypes - Magic: The Gathering (Versão Corrigida)
# Objetivo: Ingerir dados de subtypes da API do Magic: The Gathering para staging em Parquet no S3
# Características: Dados brutos, formato Parquet, tabela de referência, particionamento, incremental

# =============================================================================
# FUNÇÕES COMPARTILHADAS (AUD-08: boilerplate consolidado em ingestion_utils.py)
# =============================================================================
%run ./ingestion_utils

# =============================================================================
# CONFIGURAÇÕES GLOBAIS
# =============================================================================
API_BASE_URL = get_secret("api_base_url")
S3_BUCKET = get_secret("s3_bucket")
S3_STAGE_PREFIX = get_secret("s3_stage_prefix", "magic_the_gathering/stage")
S3_BASE_PATH = f"s3://{S3_BUCKET}/{S3_STAGE_PREFIX}"
MAX_RETRIES = int(get_secret("max_retries", "3"))

print("=" * 60)
print("CONFIGURAÇÕES PARA INGESTÃO DE SUBTYPES")
print("=" * 60)
print("API_BASE_URL: [CONFIGURADO]")
print("S3_BASE_PATH: [CONFIGURADO]")
print("Tabela de referência - sem filtro temporal")
print("=" * 60)

setup_success = setup_s3_storage(S3_BASE_PATH)
if not setup_success:
    raise Exception("Falha ao configurar S3 storage")

print("Setup concluído com sucesso")

In [ ]:
# Iniciar ingestão de subtypes
print("Iniciando ingestão de subtypes...")

subtypes_df = ingest_reference_table(
    spark, endpoint="subtypes", table_name="subtypes", field_name="subtype_name",
    api_base_url=API_BASE_URL, base_path=S3_BASE_PATH, retries=MAX_RETRIES
)

# Gerar relatório
print("=" * 50)
print("RELATÓRIO DE INGESTÃO DE SUBTYPES")
print("=" * 50)

if subtypes_df:
    print("Arquivos salvos")
else:
    print("Falha na ingestão de subtypes")

print("=" * 50)